# 01_exploration_openchargemap.ipynb

In [119]:
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
import json
from pathlib import Path

ROOT_PATH = Path.cwd().resolve().parent

load_dotenv(dotenv_path="../.env")  # cherche un fichier .env dans le dossier courant (ou parent)
cle_api = os.environ.get("OCM_API_KEY")


url = "https://api.openchargemap.io/v3/poi"

querystring = {
                "output":"json",
                "key":f"{cle_api}",
                "latitude" : 48.856614,
                "longitude" : 2.3522219,
                "distance" : 5,
                "distanceunit" : "km",
                "maxresults" : 50
            }
     
headers = {
            "Accept": "application/json",
             "User-Agent": "electric-mobility-platform/0.1 (projet portfolio Data Engineering)"}

response = requests.get(url, headers=headers, params=querystring)
print(response.status_code)

if response.status_code == 200:   
    now = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    path_target_file = ROOT_PATH / "data" / "raw" / f"{now}_paris_extract.json"
    
    with open(path_target_file, "w") as f:
        json.dump(response.json(), f)
else :
    print(f"Échec de la requête, code {response.status_code}, détail {response.text}")

200


Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?
Quels champs sont toujours présents vs parfois None ? (OperatorInfo, UsageCost, GeneralComments sont déjà None sur tes deux résultats)
Quelle est la structure d'un élément de Connections quand il n'est pas vide ? (regarde la doc de référence /v3/referencedata/ si tu veux comprendre les codes utilisés dedans, comme les types de connecteurs)

## Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?

In [120]:
resultats = response.json()

#Est-ce que Connections est vide sur toutes les entrées, ou seulement certaines ?

# Liste des connexions
connections_list = [poi["Connections"] for poi in resultats]

# Nombre de connexions vides
nb_empty_connections = len([connection for connection in connections_list if len(connection) == 0])

# Pourcentage de connexions vides
percentage_empty_connections = nb_empty_connections/len(connections_list)*100

print(f"Dans cet extract, {percentage_empty_connections} % des connections sont vides ! ")

Dans cet extract, 30.0 % des connections sont vides ! 


## Quels champs sont toujours présents vs parfois None ?

In [121]:
# Quels champs sont toujours présents vs parfois None ?

#est-ce que toutes mes entrées ont les mêmes clés, oui ou non

#Liste des clés de chaque poi (utilisation de frozenset)
frozen_keys = [frozenset(poi.keys()) for poi in resultats]

# est-ce que toutes mes entrées ont les mêmes clés, oui ou non ?
freeze_all = set(frozen_keys)
print(len(freeze_all))

1


In [122]:
# pour les clés communes, regarde combien ont une valeur None vs une vraie valeur.
keys = resultats[0].keys()

from collections import Counter

cnt_counter = Counter()

for poi in resultats:
    for key in keys:
        if poi[key] is None:
            cnt_counter[key] += 1
display(cnt_counter)

# Cas spécifique : Connections est une liste, pas juste None/valeur
connections_vides = sum(1 for poi in resultats if not poi["Connections"])
print(connections_vides)

Counter({'UserComments': 50,
         'PercentageSimilarity': 50,
         'MediaItems': 50,
         'ParentChargePointID': 50,
         'DatePlanned': 50,
         'MetadataValues': 50,
         'OperatorsReference': 43,
         'OperatorInfo': 41,
         'OperatorID': 41,
         'DateLastConfirmed': 35,
         'UsageCost': 29,
         'NumberOfPoints': 28,
         'DataProvidersReference': 23,
         'GeneralComments': 6})

15


In [130]:

for poi in resultats:
    if poi["Connections"]:
        connection = poi["Connections"]
        break

connection        

[{'ID': 76450,
  'ConnectionTypeID': 33,
  'ConnectionType': {'FormalName': 'IEC 62196-3 Configuration FF',
   'IsDiscontinued': False,
   'IsObsolete': False,
   'ID': 33,
   'Title': 'CCS (Type 2)'},
  'Reference': None,
  'StatusTypeID': 50,
  'StatusType': {'IsOperational': True,
   'IsUserSelectable': True,
   'ID': 50,
   'Title': 'Operational'},
  'LevelID': 3,
  'Level': {'Comments': '40KW and Higher',
   'IsFastChargeCapable': True,
   'ID': 3,
   'Title': 'Level 3:  High (Over 40kW)'},
  'Amps': 120,
  'Voltage': 400,
  'PowerKW': 22,
  'CurrentTypeID': 30,
  'CurrentType': {'Description': 'Direct Current', 'ID': 30, 'Title': 'DC'},
  'Quantity': None,
  'Comments': None},
 {'ID': 76451,
  'ConnectionTypeID': 2,
  'ConnectionType': {'FormalName': 'IEC 62196-3 Configuration AA',
   'IsDiscontinued': None,
   'IsObsolete': None,
   'ID': 2,
   'Title': 'CHAdeMO'},
  'Reference': None,
  'StatusTypeID': 50,
  'StatusType': {'IsOperational': True,
   'IsUserSelectable': True,
   